# Pertemuan 12 — Backtracking dan Constraint Satisfaction
N-Queens, Sudoku, pewarnaan graf. **[Modern]** CSP solver generik: forward checking, MRV, dan AC-3 — diukur dengan jumlah node yang dikunjungi.

In [ ]:
import time, itertools, random
random.seed(12)

## 1. Solver CSP generik
- `mrv=True`: pilih variabel dengan domain tersisa paling sedikit (*Minimum Remaining Values*).
- `fc=True`: *forward checking* — setelah menetapkan nilai, buang nilai konflik dari domain tetangga; jika ada domain kosong → pangkas cabang.

In [ ]:
class Limit(Exception): pass

def solve_csp(variables, domains, nbrs, ok, mrv=False, fc=False, limit=500_000):
    stats = {"nodes": 0}
    def bt(assign, doms):
        if len(assign) == len(variables): return dict(assign)
        un = [v for v in variables if v not in assign]
        v = min(un, key=lambda x: len(doms[x])) if mrv else un[0]
        for val in sorted(doms[v]):
            stats["nodes"] += 1
            if stats["nodes"] > limit: raise Limit
            if not fc and not all(ok(v, val, u, assign[u]) for u in nbrs[v] if u in assign): continue
            new = doms
            if fc:
                new, dead = dict(doms), False
                for u in nbrs[v]:
                    if u not in assign:
                        keep = {x for x in doms[u] if ok(v, val, u, x)}
                        if not keep: dead = True; break
                        new[u] = keep
                if dead: continue
            assign[v] = val
            r = bt(assign, new)
            if r: return r
            del assign[v]
        return None
    try: sol = bt({}, {v: set(d) for v, d in domains.items()})
    except Limit: sol = "LIMIT"
    return sol, stats["nodes"]

## 2. N-Queens

In [ ]:
def queens(n):
    V = list(range(n)); dom = {v: range(n) for v in V}; nb = {v: [u for u in V if u != v] for v in V}
    ok = lambda a, va, b, vb: va != vb and abs(va - vb) != abs(a - b)
    return V, dom, nb, ok
print(f"{'n':>3} | {'plain':>9} | {'MRV':>9} | {'FC':>9} | {'FC+MRV':>9}   (node dikunjungi)")
for n in [6, 8, 10, 12]:
    r = []
    for mrv, fc in [(0, 0), (1, 0), (0, 1), (1, 1)]:
        sol, nodes = solve_csp(*queens(n), mrv=bool(mrv), fc=bool(fc)); r.append(nodes if sol != "LIMIT" else ">limit")
    print(f"{n:>3} | {r[0]:>9} | {r[1]:>9} | {r[2]:>9} | {r[3]:>9}")

**Amati:** pada N-Queens semua domain awal berukuran sama, sehingga MRV saja tidak mengubah urutan (hasil = plain); MRV baru berdampak bila dipadukan dengan forward checking yang mengecilkan domain.

## 3. Sudoku sebagai CSP
Backtracking murni berurutan sering menembus batas node walau pada puzzle "mudah"; propagasi + MRV menyelesaikannya dengan puluhan–ribuan node.

In [ ]:
def sudoku_csp(grid):
    V = [(r, c) for r in range(9) for c in range(9)]
    dom = {(r, c): ({int(grid[r*9+c])} if grid[r*9+c] != "0" else set(range(1, 10))) for r, c in V}
    nb = {(r, c): [(rr, cc) for rr, cc in V if (rr, cc) != (r, c) and (rr == r or cc == c or (rr//3, cc//3) == (r//3, c//3))] for r, c in V}
    return V, dom, nb, (lambda a, va, b, vb: va != vb)
def show(sol): [print(" ".join(str(sol[(r, c)]) for c in range(9))) for r in range(9)]

mudah = "530070000600195000098000060800060003400803001700020006060000280000419005000080079"
sulit = "800000000003600000070090200050007000000045700000100030001000068008500010090000400"
for nama, puzzle in [("mudah", mudah), ("sulit", sulit)]:
    for mrv, fc in [(0, 0), (1, 1)]:
        t = time.perf_counter(); sol, nodes = solve_csp(*sudoku_csp(puzzle), mrv=bool(mrv), fc=bool(fc), limit=300_000)
        print(f"{nama:6s} MRV={mrv} FC={fc}: {'GAGAL (melebihi batas)' if sol == 'LIMIT' else 'selesai'} | node={nodes} | {time.perf_counter()-t:.2f}s")
sol, _ = solve_csp(*sudoku_csp(mudah), mrv=True, fc=True); show(sol)

## 4. AC-3 — propagasi konsistensi arc

In [ ]:
from collections import deque
def ac3(domains, nbrs, ok):
    doms = {v: set(d) for v, d in domains.items()}; q = deque((a, b) for a in doms for b in nbrs[a])
    while q:
        a, b = q.popleft(); removed = False
        for x in list(doms[a]):
            if not any(ok(a, x, b, y) for y in doms[b]): doms[a].discard(x); removed = True
        if removed:
            if not doms[a]: return None
            q.extend((c, a) for c in nbrs[a] if c != b)
    return doms
V, dom, nb, ok = sudoku_csp(mudah)
before = sum(len(d) for d in dom.values()); after_doms = ac3(dom, nb, ok); after = sum(len(d) for d in after_doms.values())
print(f"total nilai domain: sebelum AC-3 = {before}, sesudah = {after} (81 = terpecahkan hanya dengan propagasi)")

## 5. Pewarnaan peta Australia (3 warna)

In [ ]:
reg = ["WA", "NT", "SA", "Q", "NSW", "V", "T"]
edges = [("WA","NT"),("WA","SA"),("NT","SA"),("NT","Q"),("SA","Q"),("SA","NSW"),("SA","V"),("Q","NSW"),("NSW","V")]
nbm = {r: [] for r in reg}
for a, b in edges: nbm[a].append(b); nbm[b].append(a)
sol, nodes = solve_csp(reg, {r: {"merah", "hijau", "biru"} for r in reg}, nbm, lambda a, va, b, vb: va != vb, mrv=True, fc=True)
print(sol, "| node:", nodes)

## 6. Latihan / Tugas
1. Tambahkan heuristik **LCV** (nilai yang paling sedikit membatasi tetangga) dan bandingkan jumlah node.
2. Hitung **semua** solusi N-Queens untuk n = 4..10 (cocokkan dengan 2, 10, 4, 40, 92, 352, 724).
3. **Tugas:** model penjadwalan kuliah sebagai CSP (variabel = mata kuliah, domain = slot, kendala = dosen/ruang/mahasiswa bentrok) atau pewarnaan graf; bandingkan backtracking murni vs FC+MRV.

In [ ]:
def lcv_order(v, doms, nbrs, ok):
    pass  # TODO